## Step 1: Load the ML Table
Load the ML table artifact produced by Notebook 1 (one row per order).

In [1]:
import pandas as pd

# Load the artifact from Notebook 1
ml_table = pd.read_csv("artifacts/ml_table.csv")

print(f"Loaded ml_table: {ml_table.shape[0]} rows, {ml_table.shape[1]} columns")
ml_table.head()

Loaded ml_table: 99441 rows, 20 columns


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,n_items,total_price,total_freight,total_payment_value,n_payments,max_installments,review_score,n_reviews
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1.0,29.99,8.72,38.71,3.0,1.0,4.0,1.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,1.0,118.70,22.76,141.46,1.0,1.0,4.0,1.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,1.0,159.90,19.22,179.12,1.0,3.0,5.0,1.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,1.0,45.00,27.20,72.20,1.0,1.0,5.0,1.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,1.0,19.90,8.72,28.62,1.0,1.0,5.0,1.0


## Step 2: Inspect the Delivery Date Columns
Check the data types and missing values of the delivery-related columns before building the label.

In [2]:
date_cols = [
    "order_status",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

print(ml_table[date_cols].dtypes)
print()
print(ml_table[date_cols].isna().sum())

order_status                     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object

order_status                        0
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64


## Step 3: Investigate Orders Without a Delivery Date
Check the order_status for orders where order_delivered_customer_date is missing — these are likely cancelled or still in transit orders.

In [3]:
missing_delivery = ml_table[ml_table["order_delivered_customer_date"].isna()]
print(missing_delivery["order_status"].value_counts())

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64


## Step 4: Build the Late-Delivery Label
Convert date columns to datetime, then create a binary label: 1 if the order was delivered after the estimated delivery date (late), 0 if on time or early. We only build this label for orders with status "delivered" and a valid delivery date — for other statuses, the question of "late delivery" doesn't apply.

In [4]:
# Convert to datetime
ml_table["order_delivered_customer_date"] = pd.to_datetime(ml_table["order_delivered_customer_date"])
ml_table["order_estimated_delivery_date"] = pd.to_datetime(ml_table["order_estimated_delivery_date"])

# Build the label only for delivered orders with a valid delivery date
is_valid = (ml_table["order_status"] == "delivered") & (ml_table["order_delivered_customer_date"].notna())

ml_table["is_late"] = None
ml_table.loc[is_valid, "is_late"] = (
    ml_table.loc[is_valid, "order_delivered_customer_date"] > ml_table.loc[is_valid, "order_estimated_delivery_date"]
).astype(int)

print(f"Total orders: {len(ml_table)}")
print(f"Orders with a label: {ml_table['is_late'].notna().sum()}")
print(f"Orders without a label: {ml_table['is_late'].isna().sum()}")

Total orders: 99441
Orders with a label: 96470
Orders without a label: 2971


## Step 5: Sanity-Check the Label
Verify the label is correct by manually inspecting a few real orders — comparing the actual delivery date, the estimated delivery date, and the resulting label.

In [5]:
check_cols = [
    "order_id",
    "order_status",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "is_late"
]

# A few late examples
print("Examples labeled as LATE (1):")
print(ml_table[ml_table["is_late"] == 1][check_cols].head(3))

print("\nExamples labeled as ON-TIME (0):")
print(ml_table[ml_table["is_late"] == 0][check_cols].head(3))

Examples labeled as LATE (1):
                            order_id order_status  \
20  203096f03d82e0dffbc41ebc2e2bcfb7    delivered   
25  fbf9ac61453ac646ce8ad9783d7d0af6    delivered   
35  8563039e855156e48fccee4d611a3196    delivered   

   order_delivered_customer_date order_estimated_delivery_date is_late  
20           2017-10-09 22:23:46                    2017-09-28       1  
25           2018-03-21 22:03:54                    2018-03-12       1  
35           2018-03-20 00:59:25                    2018-03-20       1  

Examples labeled as ON-TIME (0):
                           order_id order_status  \
0  e481f51cbdc54678b7cc49136f2d6af7    delivered   
1  53cdb2fc8bc7dce0b6741e2150273451    delivered   
2  47770eb9100c2d0c44946d9cf07ec65d    delivered   

  order_delivered_customer_date order_estimated_delivery_date is_late  
0           2017-10-10 21:25:13                    2017-10-18       0  
1           2018-08-07 15:27:45                    2018-08-13       0  
2     

## Step 6: Class Distribution
Check how many orders are late vs on-time, and whether we have a class imbalance problem.

In [6]:
label_counts = ml_table["is_late"].value_counts(dropna=True)
label_pct = ml_table["is_late"].value_counts(normalize=True, dropna=True) * 100

print("Counts:")
print(label_counts)
print("\nPercentage:")
print(label_pct.round(2))

Counts:
is_late
0    88644
1     7826
Name: count, dtype: int64

Percentage:
is_late
0    91.89
1     8.11
Name: proportion, dtype: float64


## Step 7: Save the Labeled Table
Save the ML table with the new `is_late` label as an artifact for the next notebook (train/validation/test split).

**Class imbalance note:** The label is imbalanced — about 92% on-time vs 8% late. This will need to be handled when choosing the evaluation metric and training approach in later notebooks.

In [7]:
ml_table.to_csv("artifacts/labeled_table.csv", index=False)
print(f"✅ Saved labeled_table.csv with {ml_table.shape[0]} rows and {ml_table.shape[1]} columns")

✅ Saved labeled_table.csv with 99441 rows and 21 columns
